# Lab 1: NLPScholar practice
### COSC 426: Fall 2025, Colgate University

Use this notebook to answer the questions in `Lab1.md`. For answers where you are looking at data files generated by the `evaluate` or `analyze` modes (which is almost all of them), you should load the data into the ipynb notebook and proceed from there. 

## Part 1: Minimal Pair

In [15]:
# Load dataframe from tsv file
import pandas as pd

df1 = pd.read_csv('results/agreement_1.tsv', sep='\t')

# print diff when lemma is LIKE
print(df1[df1['lemma'] == 'LIKE'].to_string(index=False))

 Unnamed: 0                   model lemma  acc      diff  expected  unexpected  macrodiff
          1 distilbert-base-uncased  LIKE 1.00 -2.278639  6.961845    9.240484  -2.278639
          3   distilbert/distilgpt2  LIKE 0.75 -1.516558  8.794097   10.310654  -1.516558
          5                    gpt2  LIKE 0.75 -1.550561  8.680158   10.230719  -1.550561


1.  The difference for `gpt2` is -1.5505609698593612, for `distilbert/distilgpt2` is -1.516557903587818, and for `distilbert-base-uncased` is -2.2786386686751943.

In [16]:
df2 = pd.read_csv('results/agreement_2.tsv', sep='\t')

# Group by model and compare singular vs plural
comparison = []
for model, group in df2.groupby("model"):
    sing = group.loc[group["number"] == "sing", "diff"].values[0]
    plu = group.loc[group["number"] == "plu", "diff"].values[0]
    comparison.append({
        "model": model,
        "sing_diff": sing,
        "plu_diff": plu,
        "greater_for": "singular" if sing > plu else "plural"
    })

result = pd.DataFrame(comparison)
print("2.   ")
print(result)
print("All of them have greater singular difference.")

2.   
                     model  sing_diff  plu_diff greater_for
0  distilbert-base-uncased  -2.329134 -4.133037    singular
1    distilbert/distilgpt2  -1.433037 -3.141553    singular
2                     gpt2  -1.607004 -3.184358    singular
All of them have greater singular difference.


In [102]:
input_file = "data/agreement.tsv"
output_file = "data/agreement_3.tsv"

# read the TSV file
df = pd.read_csv(input_file, sep="\t")

# drop the ROI column
if "ROI" in df.columns:
    df = df.drop(columns=["ROI"])

# save to new file
df.to_csv(output_file, sep="\t", index=False)

In [17]:
df3 = pd.read_csv('results/agreement_3.tsv', sep='\t')

comparison = []
for model, group in df3.groupby("model"):
    sing = group.loc[group["number"] == "sing", "diff"].values[0]
    plu = group.loc[group["number"] == "plu", "diff"].values[0]
    comparison.append({
        "model": model,
        "sing_diff": sing,
        "plu_diff": plu,
        "greater_for": "singular" if sing > plu else "plural"
    })

result = pd.DataFrame(comparison)
print("3.   ")
print(result)

3.   
                     model  sing_diff  plu_diff greater_for
0  distilbert-base-uncased  -1.291045 -2.580855    singular
1    distilbert/distilgpt2  -0.481563 -1.570776    singular
2                     gpt2  -0.610404 -1.592179    singular


In [18]:
df4 = pd.read_csv('results/agreement_4.tsv', sep='\t')
print("4.   ")
print(df4[["model", "expected"]])

4.   
                     model  expected
0  distilbert-base-uncased  0.089119
1    distilbert/distilgpt2  0.171080
2                     gpt2  0.184998


In [20]:
df5_ignore = pd.read_csv('results/agreement_5_1_byword.tsv', sep='\t')
df5_previous = pd.read_csv('results/agreement_5_2_byword.tsv', sep='\t')

# Keep only rows where comparison is "expected"
df5_ignore = df5_ignore[df5_ignore["comparison"] == "expected"]
df5_previous = df5_previous[df5_previous["comparison"] == "expected"]

# Filter the rows where the token is "chameleons" (no period) or "chameleons." (with period)
df5_ignore = df5_ignore[df5_ignore["word_mod"] == "chameleons"]
df5_previous = df5_previous[df5_previous["word_mod"] == "chameleons"]

# Group by model and compute mean prob for each
mean_ignore = df5_ignore.groupby("model")["prob"].mean().reset_index(name="mean_prob_ignore")
mean_previous = df5_previous.groupby("model")["prob"].mean().reset_index(name="mean_prob_previous")

mean_probs = pd.merge(mean_ignore, mean_previous, on="model")
print("5.   ")
print(mean_probs)


5.   
                     model  mean_prob_ignore  mean_prob_previous
0  distilbert-base-uncased          0.319130            0.318630
1    distilbert/distilgpt2          0.191020            0.123809
2                     gpt2          0.251116            0.155670


6. `minimal_pairs_bad.tsv` has inconsistent and incorrect assignment of the pairid for the last four sentences. This doesn't affect the `evaluate` mode. However, in the `analyze` mode it ignores the last 2 pair of sentences.

## Part 2: Token Classification

In [55]:
dat1 = pd.read_csv('results/ner_1_bycond.tsv', sep='\t')

acc = dat1['accuracy'].iloc[0]
print(f"1.  Overall accuracy of the model (ignore punctutation): {acc*100:.4f}%")

dat1_1 = pd.read_csv('results/ner_1_1_bycond.tsv', sep='\t')

acc = dat1_1['accuracy'].iloc[0]
print(f"Overall accuracy of the model (keep punctutation): {acc*100:.4f}%")

1.  Overall accuracy of the model (ignore punctutation): 2.1739%
Overall accuracy of the model (keep punctutation): 34.5455%


In [56]:
dat2 = pd.read_csv('results/ner_2_bycond.tsv', sep='\t')

acc = dat2['accuracy'].iloc[0]
print(f"2.  Overall accuracy of the model ignore the O label (ignore punctutation): {acc*100:.4f}%")


2.  Overall accuracy of the model ignore the O label (ignore punctutation): 8.0000%


In [57]:
dat3 = pd.read_csv('results/ner_3_bycond.tsv', sep='\t')

acc = dat3['accuracy'].iloc[0]
print(f"3.  Overall accuracy of the model ignore the O label (keep punctutation): {acc*100:.4f}%")

3.  Overall accuracy of the model ignore the O label (keep punctutation): 0.0000%


In [58]:
dat4 = pd.read_csv('results/ner_4_bytokentype.tsv', sep='\t')

b_location_data = dat4[dat4['target_class'] == 'B-location']

full_sentences = b_location_data[b_location_data['condition'] == 'full']
short_sentences = b_location_data[b_location_data['condition'] == 'short']

full_metrics = {
    'condition': 'full',
    'precision': full_sentences['precision'].item(),
    'recall': full_sentences['recall'].item(),
    'f1': full_sentences['f1'].item()
}

short_metrics = {
    'condition': 'short', 
    'precision': short_sentences['precision'].item(),
    'recall': short_sentences['recall'].item(),
    'f1': short_sentences['f1'].item()
}

b_location_comparison = pd.DataFrame([full_metrics, short_metrics])

print("4.   B-location accuracy by sentence length:")
print(b_location_comparison)

4.   B-location accuracy by sentence length:
  condition  precision    recall        f1
0      full        0.5  0.400000  0.444444
1     short        0.5  0.666667  0.571429


5.  We need to change the model to `bert-base-NER` and also update `id2label` section with 
``` yaml
id2label:
  0: O
  1: B-MISC
  2: I-MISC
  3: B-PER
  4: I-PER
  5: B-ORG
  6: I-ORG
  7: B-LOC
  8: I-LOC
```

## Part 3: Text Classification

In [113]:
data1 = pd.read_csv('results/sentiment_1.tsv', sep='\t')

trained_model = data1[data1['model'] == 'siebert/sentiment-roberta-large-english']
untrained_model = data1[data1['model'] == 'roberta-large']

trained_accuracy = trained_model['accuracy'].item()
untrained_accuracy = untrained_model['accuracy'].item()

# data1_trained = pd.read_csv('results/sentiment_1_1.tsv', sep='\t')
# data1_untrained = pd.read_csv('results/sentiment_1_2.tsv', sep='\t')

# trained_accuracy = data1_trained['accuracy'].item()
# untrained_accuracy = data1_untrained['accuracy'].item()

print("1.   ")
print(f"   Trained model (siebert/sentiment-roberta-large-english): {trained_accuracy*100:.4f}%")
print(f"   Untrained model (roberta-large): {untrained_accuracy*100:.4f}%")
print(f"   Answer: {'Yes' if trained_accuracy > untrained_accuracy else 'No'}, the trained model {'has higher' if trained_accuracy > untrained_accuracy else 'does not have higher'} accuracy")


1.   
   Trained model (siebert/sentiment-roberta-large-english): 87.5000%
   Untrained model (roberta-large): 50.0000%
   Answer: Yes, the trained model has higher accuracy


In [114]:
data2 = pd.read_csv('results/sentiment_2.tsv', sep='\t')

trained_full = data2[(data2['model'] == 'siebert/sentiment-roberta-large-english') & (data2['type'] == 'Full')]['accuracy'].item()
trained_oneline = data2[(data2['model'] == 'siebert/sentiment-roberta-large-english') & (data2['type'] == 'One-line')]['accuracy'].item()
untrained_full = data2[(data2['model'] == 'roberta-large') & (data2['type'] == 'Full')]['accuracy'].item()
untrained_oneline = data2[(data2['model'] == 'roberta-large') & (data2['type'] == 'One-line')]['accuracy'].item()

difference_full = trained_full - untrained_full
difference_oneline = trained_oneline - untrained_oneline

print("2.   ")
print(f"   Full reviews - Trained: {trained_full*100:.4f}%, Untrained: {untrained_full*100:.4f}%")
print(f"   One-line reviews - Trained: {trained_oneline*100:.4f}%, Untrained: {untrained_oneline*100:.4f}%")
print(f"   Difference for Full reviews: {difference_full*100:.4f}%")
print(f"   Difference for One-line reviews: {difference_oneline*100:.4f}%")

# data = pd.read_csv('data/sentiment.tsv', sep='\t')
# data2_pred = pd.read_csv('predictions/sentiment_2.tsv', sep='\t')

# data2_pred = data2_pred.merge(data[["textid", "type"]], on="textid", how="left")

# trained_full = data2_pred[(data2_pred['model'] == 'siebert/sentiment-roberta-large-english') & (data2_pred['type'] == 'Full')]
# trained_full_accuracy = (trained_full['target'] == trained_full['predicted']).mean()
# trained_oneline = data2_pred[(data2_pred['model'] == 'siebert/sentiment-roberta-large-english') & (data2_pred['type'] == 'One-line')]
# trained_oneline_accuracy = (trained_oneline['target'] == trained_oneline['predicted']).mean()
# untrained_full = data2_pred[(data2_pred['model'] == 'roberta-large') & (data2_pred['type'] == 'Full')]
# untrained_full_accuracy = (untrained_full['target'] == untrained_full['predicted']).mean()
# untrained_oneline = data2_pred[(data2_pred['model'] == 'roberta-large') & (data2_pred['type'] == 'One-line')]
# untrained_oneline_accuracy = (untrained_oneline['target'] == untrained_oneline['predicted']).mean()

# difference_full = trained_full_accuracy - untrained_full_accuracy
# difference_oneline = trained_oneline_accuracy - untrained_oneline_accuracy

# print("2.   ")
# print(f"   Full reviews - Trained: {trained_full_accuracy*100:.4f}%, Untrained: {untrained_full_accuracy*100:.4f}%")
# print(f"   One-line reviews - Trained: {trained_oneline_accuracy*100:.4f}%, Untrained: {untrained_oneline_accuracy*100:.4f}%")
# print(f"   Difference for Full reviews: {difference_full*100:.4f}%")
# print(f"   Difference for One-line reviews: {difference_oneline*100:.4f}%")
# print(f"   Answer: {'Yes' if difference_full > difference_oneline else 'No'}, the trained model {'has a greater difference for Full reviews' if difference_full > difference_oneline else 'does not have a greater difference for Full reviews'}")

2.   
   Full reviews - Trained: 100.0000%, Untrained: 50.0000%
   One-line reviews - Trained: 75.0000%, Untrained: 50.0000%
   Difference for Full reviews: 50.0000%
   Difference for One-line reviews: 25.0000%


In [115]:
data3 = pd.read_csv('results/sentiment_3_bytarget.tsv', sep='\t')

positive_reviews = data3[data3['target_class'] == 'Positive']
negative_reviews = data3[data3['target_class'] == 'Negative']

positive_f1 = positive_reviews['f1'].mean()
negative_f1 = negative_reviews['f1'].mean()

print("3.   ")
print(f"   Average F1 score for positive reviews: {positive_f1:.4f}")
print(f"   Average F1 score for negative reviews: {negative_f1:.4f}")
print(f"   Answer: {'Yes' if positive_f1 > negative_f1 else 'No'}, {'positive' if positive_f1 > negative_f1 else 'negative'} reviews have higher F1 score")

3.   
   Average F1 score for positive reviews: 0.7778
   Average F1 score for negative reviews: 0.4286
   Answer: Yes, positive reviews have higher F1 score


In [116]:
input_file = "data/sentiment.tsv"
output_file = "data/sentiment_chatgpt.tsv"

# Read the TSV file
df = pd.read_csv(input_file, sep="\t")

# Keep only rows where source == "ChatGPT"
if "source" in df.columns:
    df = df[df["source"] == "ChatGPT"]

# Save to new file
df.to_csv(output_file, sep="\t", index=False)


In [117]:
data4 = pd.read_csv('results/sentiment_4_bytarget.tsv', sep='\t')

positive_reviews = data4[data4['target_class'] == 'Positive']
negative_reviews = data4[data4['target_class'] == 'Negative']

positive_f1 = positive_reviews['f1'].mean()
negative_f1 = negative_reviews['f1'].mean()

print("4.   ")
print(f"   Average F1 score for positive reviews: {positive_f1:.4f}")
print(f"   Average F1 score for negative reviews: {negative_f1:.4f}")
print(f"   Answer: {'Positive' if positive_f1 > negative_f1 else 'Negative'} reviews have higher F1 score when considering just the ChatGPT generated reviews")

4.   
   Average F1 score for positive reviews: 0.7333
   Average F1 score for negative reviews: 0.3333
   Answer: Positive reviews have higher F1 score when considering just the ChatGPT generated reviews


In [122]:
data5 = pd.read_csv('predictions/sentiment_1.tsv', sep='\t')

avg_prob = data5.groupby("model")["prob"].mean().reset_index(name="prob")

print("5.   Average probability of the predicted label for each model:")
for _, row in avg_prob.iterrows():
    print(f"   {row['model']}: {row['prob']:.4f}")

5.   Average probability of the predicted label for each model:
   roberta-large: 0.5454
   siebert/sentiment-roberta-large-english: 0.9990
